In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from gensim.utils import simple_preprocess

# ========================
# 1. Load data
# ========================

# Load query dari Excel
query_dataframe = pd.read_excel('step6_final.xlsx')  # Sesuaikan nama file
query_data = query_dataframe['hasil_preprocessing'].dropna().tolist()

# Korpus: daftar isu pemilu
corpus_data = [
          'Adanya kekerasan/kerusuhan berbasis SARA dalam isu-isu politik dan kepemiluan.',
          'Adanya kekerasan/kerusuhan yang melibatkan tokoh publik/politik/aparat keamanan.',
          'Adanya bencana alam yang mengganggu tahapan Pemilu/Pilkada.',
          'Adanya bencana non alam (pandemi covid 19) yang mengganggu tahapan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Peserta Pemilu dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Penyelenggara Pemilu dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Pemilih dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya perusakan fasilitas penyelenggaraan Pemilu.',
          'Adanya putusan DKPP yang ditujukan kepada Jajaran KPU dan/atau Bawaslu.',
          'Adanya Rekomendasi dan/atau Putusan Bawaslu yang tidak ditindaklanjuti oleh KPU.',
          'Adanya rekomendasi Bawaslu terkait dengan perubahan suara pada proses rekapitulasi suara.',
          'Adanya Rekomendasi Bawaslu terkait ketidaknetralan ASN/TNI/POLRI.',
          'Adanya penyalahgunaan anggaran Pemilu/Pilkada.',
          'Adanya intimidasi kepada penyelenggara pemilu yang dilakukan oleh Kepala Daerah.',
          'Adanya himbauan dan/atau tindakan untuk memilih calon tertentu dari pemerintah lokal.',
          'Adanya himbauan dan/atau tindakan untuk menolak calon tertentu dari pemerintah lokal atau tokoh masyarakat.',
          'Adanya pemilih memenuhi syarat tetapi tidak terdaftar dalam daftar pemilih tetap.',
          'Adanya pemilih tidak memenuhi syarat terdaftar dalam pemilih tetap.',
          'Adanya penduduk potensial memilih tetapi tidak memiliki KTP-Elektronik.',
          'Adanya pemilih ganda dalam daftar pemilih (logistik).',
          'Adanya penggelembungan jumlah data pemilih.',
          'Adanya Pemilih Pindah Memilih (DPTB) yang Tidak Dapat Memberikan Hak Suaranya.',
          'Adanya ketidakprofesionalan Penyelenggara Pemilu yang merugikan kampanye calon.',
          'Adanya kampanye yang difasilitasi KPU tidak sesuai dengan ketentuan.',
          'Adanya penyelenggara Pemilu yang menunjukan sikap keberpihakan dalam tahapan kampanye.',
          'Adanya penyelenggara Pemilu yang melakukan pengabaian terhadap protokol kesehatan dalam kegiatan kampanye.',
          'Adanya perlengkapan pemungutan suara yang tidak sesuai dengan ketentuan Ada keterlambatan perlengkapan (logistik) pemungutan suara.',
          'Ada keterlambatan perlengkapan (logistik) pemungutan suara.',
          'Adanya logistik berupa surat suara pemungutan suara yang tertukar.',
          'Adanya informasi tidak terdapat TPS untuk menfasilitasi pemungutan suara bagi pemilih di LAPAS/RS/SEKOLAH DINAS/.',
          'Adanya informasi pelanggaraan saat pemungutan suara di Pemilu/Pilkada.',
          'Adanya pemungutan suara ulang di Pemilu/Pilkada.',
          'Adanya penghitungan suara ulang di Pemilu/Pilkada.',
          'Adanya Pemungutan suara susulan di Pemilu/Pilkada.',
          'Adanya saran perbaikan (catatan khusus) dari pengawas saat pemungutan suara.',
          'Adanya komplain dari saksi saat pemungutan atau penghitungan suara.',
          'Adanya gugatan atas hasil Pemilu/Pilkada.',
          'Adanya keberatan dan/atau sengketa proses Pemilu/Pilkada.',
          'Adanya TPS yang tidak dihadiri oleh Pengawas TPS pada Pemilu/Pilkada.',
          'Adanya Informasi ketiadaan Pengawas Pemilu Tingkat Kelurahan/Desa.',
          'Adanya Penolakan terhadap calon perempuan di Pemilu/Pilkada.',
          'Adanya Penolakan terhadap calon berlatarbelakang etnis, Suku, dan Agama tertentu di Pemilu/Pilkada.',
          'Adanya dokumen palsu dalam proses pencalonan di Pemilu dan Pilkada.',
          'Adanya informasi kampanye di luar jadwal yang dilakukan oleh peserta Pemilu.',
          'Adanya iklan kampanye di luar jadwal yang dilakukan oleh peserta Pemilu.',
          'Adanya materi kampanye yang bermuatan SARA di tempat umum.',
          'Adanya materi kampanye yang bermuata SARA di sosial media akun lokal (Grup WA / Facebook).',
          'Adanya materi kampanye Hoax di tempat umum.',
          'Adanya materi kampanye Hoax di sosial media (Grup WA / Facebook).',
          'Adanya materi kampanye ujaran kebencian di tempat umum.',
          'Adanya materi kampanye ujaran kebencian di sosial media (Grup WA / Facebook).',
          'Adanya pelanggaran lokasi kampanye yang dilakukan oleh peserta Pemilu/Pilkada.',
          'Adanya konflik antar pendukung peserta pemilu/pasangan calon.',
          'Adanya Laporan tentang Politik Uang yang dilakukan oleh Peserta/Tim Sukses/Tim Kampanye Pemilu.',
          'Adanya peserta pemilu/calon yang tidak melaporkan dana kampanye.',
          'Adanya upaya untuk menghalanghalangi Pemilih memberikan suara di TPS.',
          'Adanya mobilisasi pemilih tambahan secara mendadak.',
          'Adanya pemilih tambahan melebihi jumlah surat suara cadangan 2%.',
          'Adanya laporan masyarakat yang tidak ditindaklanjuti oleh Pengawas Pemilu.',
          'Adanya Informasi tentang ketiadaan pemantau Pemilu.',
          'Adanya mobilisasi penolakan terhadap pelaksanaan pemilu/pilkada dari kelompok masyarakat.'
]

# ========================
# 2. Load IndoBERT
# ========================

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# ========================
# 3. Fungsi bantu
# ========================

def get_sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden_state = outputs.last_hidden_state
        attention_mask = inputs['attention_mask']

        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
        sum_mask = mask_expanded.sum(1)
        embedding = sum_embeddings / sum_mask
        
    return embedding.cpu().numpy()[0]

# ========================
# 4. Embedding semua kalimat
# ========================

print("Mengubah query menjadi vektor...")
query_embeddings = [get_sentence_embedding(text) for text in query_data]

print("Mengubah korpus menjadi vektor...")
corpus_embeddings = [get_sentence_embedding(text) for text in corpus_data]

# ========================
# 5. Hitung kemiripan
# ========================

results = []

for i, query_emb in enumerate(query_embeddings):
    similarities = cosine_similarity([query_emb], corpus_embeddings)[0]
    top_idx = np.argmax(similarities)
    least_idx = np.argmin(similarities)
    
    results.append({
        "Query_Index": i + 1,
        "Query": query_data[i],
        "Top_Doc_Index": top_idx + 1,
        "Top_Document": corpus_data[top_idx],
        "Similarity": similarities[top_idx],
        "Type": "paling Mirip"
    })

# ========================
# 6. Simpan ke Excel
# ========================

df_result = pd.DataFrame(results)
df_result.to_excel("hasil_kemiripan_indobert.xlsx", index=False)

print("Proses selesai. Hasil disimpan di 'hasil_kemiripan_indobert-ASLI.xlsx'.")


Mengubah query menjadi vektor...
Mengubah korpus menjadi vektor...
Proses selesai. Hasil disimpan di 'hasil_kemiripan_indobert-ASLI.xlsx'.


In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from gensim.utils import simple_preprocess

# ========================
# 1. Load data
# ========================

# Load query dari Excel
query_dataframe = pd.read_excel('step6_final.xlsx')  # Sesuaikan nama file
query_data = query_dataframe['hasil_preprocessing'].dropna().tolist()

# Korpus: daftar isu pemilu
corpus_data = [
          'Adanya kekerasan/kerusuhan berbasis SARA dalam isu-isu politik dan kepemiluan.',
          'Adanya kekerasan/kerusuhan yang melibatkan tokoh publik/politik/aparat keamanan.',
          'Adanya bencana alam yang mengganggu tahapan Pemilu/Pilkada.',
          'Adanya bencana non alam (pandemi covid 19) yang mengganggu tahapan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Peserta Pemilu dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Penyelenggara Pemilu dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya Intimidasi Terhadap Pemilih dalam proses pelaksanaan Pemilu/Pilkada.',
          'Adanya perusakan fasilitas penyelenggaraan Pemilu.',
          'Adanya putusan DKPP yang ditujukan kepada Jajaran KPU dan/atau Bawaslu.',
          'Adanya Rekomendasi dan/atau Putusan Bawaslu yang tidak ditindaklanjuti oleh KPU.',
          'Adanya rekomendasi Bawaslu terkait dengan perubahan suara pada proses rekapitulasi suara.',
          'Adanya Rekomendasi Bawaslu terkait ketidaknetralan ASN/TNI/POLRI.',
          'Adanya penyalahgunaan anggaran Pemilu/Pilkada.',
          'Adanya intimidasi kepada penyelenggara pemilu yang dilakukan oleh Kepala Daerah.',
          'Adanya himbauan dan/atau tindakan untuk memilih calon tertentu dari pemerintah lokal.',
          'Adanya himbauan dan/atau tindakan untuk menolak calon tertentu dari pemerintah lokal atau tokoh masyarakat.',
          'Adanya pemilih memenuhi syarat tetapi tidak terdaftar dalam daftar pemilih tetap.',
          'Adanya pemilih tidak memenuhi syarat terdaftar dalam pemilih tetap.',
          'Adanya penduduk potensial memilih tetapi tidak memiliki KTP-Elektronik.',
          'Adanya pemilih ganda dalam daftar pemilih (logistik).',
          'Adanya penggelembungan jumlah data pemilih.',
          'Adanya Pemilih Pindah Memilih (DPTB) yang Tidak Dapat Memberikan Hak Suaranya.',
          'Adanya ketidakprofesionalan Penyelenggara Pemilu yang merugikan kampanye calon.',
          'Adanya kampanye yang difasilitasi KPU tidak sesuai dengan ketentuan.',
          'Adanya penyelenggara Pemilu yang menunjukan sikap keberpihakan dalam tahapan kampanye.',
          'Adanya penyelenggara Pemilu yang melakukan pengabaian terhadap protokol kesehatan dalam kegiatan kampanye.',
          'Adanya perlengkapan pemungutan suara yang tidak sesuai dengan ketentuan Ada keterlambatan perlengkapan (logistik) pemungutan suara.',
          'Ada keterlambatan perlengkapan (logistik) pemungutan suara.',
          'Adanya logistik berupa surat suara pemungutan suara yang tertukar.',
          'Adanya informasi tidak terdapat TPS untuk menfasilitasi pemungutan suara bagi pemilih di LAPAS/RS/SEKOLAH DINAS/.',
          'Adanya informasi pelanggaraan saat pemungutan suara di Pemilu/Pilkada.',
          'Adanya pemungutan suara ulang di Pemilu/Pilkada.',
          'Adanya penghitungan suara ulang di Pemilu/Pilkada.',
          'Adanya Pemungutan suara susulan di Pemilu/Pilkada.',
          'Adanya saran perbaikan (catatan khusus) dari pengawas saat pemungutan suara.',
          'Adanya komplain dari saksi saat pemungutan atau penghitungan suara.',
          'Adanya gugatan atas hasil Pemilu/Pilkada.',
          'Adanya keberatan dan/atau sengketa proses Pemilu/Pilkada.',
          'Adanya TPS yang tidak dihadiri oleh Pengawas TPS pada Pemilu/Pilkada.',
          'Adanya Informasi ketiadaan Pengawas Pemilu Tingkat Kelurahan/Desa.',
          'Adanya Penolakan terhadap calon perempuan di Pemilu/Pilkada.',
          'Adanya Penolakan terhadap calon berlatarbelakang etnis, Suku, dan Agama tertentu di Pemilu/Pilkada.',
          'Adanya dokumen palsu dalam proses pencalonan di Pemilu dan Pilkada.',
          'Adanya informasi kampanye di luar jadwal yang dilakukan oleh peserta Pemilu.',
          'Adanya iklan kampanye di luar jadwal yang dilakukan oleh peserta Pemilu.',
          'Adanya materi kampanye yang bermuatan SARA di tempat umum.',
          'Adanya materi kampanye yang bermuata SARA di sosial media akun lokal (Grup WA / Facebook).',
          'Adanya materi kampanye Hoax di tempat umum.',
          'Adanya materi kampanye Hoax di sosial media (Grup WA / Facebook).',
          'Adanya materi kampanye ujaran kebencian di tempat umum.',
          'Adanya materi kampanye ujaran kebencian di sosial media (Grup WA / Facebook).',
          'Adanya pelanggaran lokasi kampanye yang dilakukan oleh peserta Pemilu/Pilkada.',
          'Adanya konflik antar pendukung peserta pemilu/pasangan calon.',
          'Adanya Laporan tentang Politik Uang yang dilakukan oleh Peserta/Tim Sukses/Tim Kampanye Pemilu.',
          'Adanya peserta pemilu/calon yang tidak melaporkan dana kampanye.',
          'Adanya upaya untuk menghalanghalangi Pemilih memberikan suara di TPS.',
          'Adanya mobilisasi pemilih tambahan secara mendadak.',
          'Adanya pemilih tambahan melebihi jumlah surat suara cadangan 2%.',
          'Adanya laporan masyarakat yang tidak ditindaklanjuti oleh Pengawas Pemilu.',
          'Adanya Informasi tentang ketiadaan pemantau Pemilu.',
          'Adanya mobilisasi penolakan terhadap pelaksanaan pemilu/pilkada dari kelompok masyarakat.'
]

# ========================
# 2. Load IndoBERT
# ========================

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# ========================
# 3. Fungsi bantu
# ========================

def get_sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden_state = outputs.last_hidden_state
        attention_mask = inputs['attention_mask']

        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
        sum_mask = mask_expanded.sum(1)
        embedding = sum_embeddings / sum_mask
        
    return embedding.cpu().numpy()[0]

# ========================
# 4. Embedding semua kalimat
# ========================

print("Mengubah query menjadi vektor...")
query_embeddings = [get_sentence_embedding(text) for text in query_data]

print("Mengubah korpus menjadi vektor...")
corpus_embeddings = [get_sentence_embedding(text) for text in corpus_data]

# ========================
# 5. Hitung kemiripan
# ========================

results = []

for i, query_emb in enumerate(query_embeddings):
    similarities = cosine_similarity([query_emb], corpus_embeddings)[0]
    top_idx = np.argmax(similarities)
    least_idx = np.argmin(similarities)
    
    results.append({
        "Query_Index": i + 1,
        "Query": query_data[i],
        "Top_Doc_Index": top_idx + 1,
        "Top_Document": corpus_data[top_idx],
        "Similarity": similarities[top_idx],
        "Type": "paling Mirip"
    })
    
    results.append({
        "Query_Index": i + 1,
        "Query": query_data[i],
        "Top_Doc_Index": least_idx + 1,
        "Top_Document": corpus_data[least_idx],
        "Similarity": similarities[least_idx],
        "Type": "paling tidak mirip"
    })

# ========================
# 6. Simpan ke Excel
# ========================

df_result = pd.DataFrame(results)
df_result.to_excel("haindobert_miripDANtidakmirip.xlsx", index=False)

print("Proses selesai. Hasil disimpan di 'hasil_kemiripan_indobert.xlsx'.")


Mengubah query menjadi vektor...
Mengubah korpus menjadi vektor...
Proses selesai. Hasil disimpan di 'hasil_kemiripan_indobert.xlsx'.
